In [ ]:
import os
import numpy as np
import pandas as pd
import random
import seaborn as sns
from matplotlib import pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dropout
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense
from sklearn.preprocessing import MinMaxScaler

# Reproducibility
np.random.seed(42)
random.seed(42)
tf.random.set_seed(42)

# ============================================================================
# STEP 1: Load Raw Data
# ============================================================================

print("="*80)
print("STEP 1: LOADING RAW DATA")
print("="*80)

DATA_PATH = os.path.join('V_0.csv')
dataframe = pd.read_csv(DATA_PATH)

# Datetime processing
dataframe['datetimeCST'] = pd.to_datetime(dataframe['datetimeCST'])

# Select required columns
df_raw = dataframe[['datetimeCST', 'Hz']].copy()
df_raw.set_index('datetimeCST', inplace=True)

print(f"Total rows loaded: {len(df_raw):,}")
print(f"Date range: {df_raw.index.min()} to {df_raw.index.max()}")
print(f"Missing values: {df_raw['Hz'].isna().sum()}")
print("\nBasic Statistics (Raw Data):")
print(df_raw['Hz'].describe())

# ============================================================================
# STEP 2: Analyze Raw Data
# ============================================================================

print("\n" + "="*80)
print("STEP 2: ANALYZING RAW DATA")
print("="*80)

# Check for zeros
zero_count = (df_raw['Hz'] == 0).sum()
zero_percentage = (zero_count / len(df_raw)) * 100

print(f"Zero values: {zero_count:,} ({zero_percentage:.2f}%)")
print(f"Values < 55 Hz: {(df_raw['Hz'] < 55).sum():,}")
print(f"Values >= 55 Hz: {(df_raw['Hz'] >= 55).sum():,}")

# Value distribution
print("\nValue Ranges:")
print(f"  Hz = 0: {(df_raw['Hz'] == 0).sum():,}")
print(f"  0 < Hz < 55: {((df_raw['Hz'] > 0) & (df_raw['Hz'] < 55)).sum():,}")
print(f"  Hz >= 55: {(df_raw['Hz'] >= 55).sum():,}")

# ============================================================================
# STEP 3: Identify Short Outages (< 10 minutes)
# ============================================================================

print("\n" + "="*80)
print("STEP 3: IDENTIFYING SHORT OUTAGES")
print("="*80)

df_processed = df_raw.copy()

# Identify zero periods
is_zero = df_processed['Hz'] == 0
zero_groups = (is_zero != is_zero.shift()).cumsum()

# Calculate duration of each zero period
zero_durations = []
for group_id in zero_groups[is_zero].unique():
    group_data = df_processed[zero_groups == group_id]
    if len(group_data) > 0 and group_data['Hz'].iloc[0] == 0:
        duration_minutes = (group_data.index[-1] - group_data.index[0]).total_seconds() / 60
        zero_durations.append({
            'start': group_data.index[0],
            'end': group_data.index[-1],
            'duration_minutes': duration_minutes,
            'count': len(group_data)
        })

zero_df = pd.DataFrame(zero_durations)

if len(zero_df) > 0:
    short_outages = zero_df[zero_df['duration_minutes'] < 10]
    long_outages = zero_df[zero_df['duration_minutes'] >= 10]
    
    print(f"Total zero periods found: {len(zero_df)}")
    print(f"Short outages (< 10 min): {len(short_outages)}")
    print(f"Long outages (>= 10 min): {len(long_outages)}")
    
    if len(short_outages) > 0:
        print(f"\nShort outage statistics:")
        print(f"  Total duration: {short_outages['duration_minutes'].sum():.2f} minutes")
        print(f"  Average duration: {short_outages['duration_minutes'].mean():.2f} minutes")
        print(f"  Max duration: {short_outages['duration_minutes'].max():.2f} minutes")
else:
    print("No zero periods found in data")
    short_outages = pd.DataFrame()

# ============================================================================
# STEP 4: Fill Short Outages with Interpolation
# ============================================================================

print("\n" + "="*80)
print("STEP 4: FILLING SHORT OUTAGES")
print("="*80)

points_filled = 0

if len(short_outages) > 0:
    for idx, outage in short_outages.iterrows():
        # Get the mask for this outage period
        mask = (df_processed.index >= outage['start']) & (df_processed.index <= outage['end'])
        
        # Mark as NaN for interpolation
        df_processed.loc[mask, 'Hz'] = np.nan
        points_filled += mask.sum()
    
    # Interpolate missing values
    df_processed['Hz'] = df_processed['Hz'].interpolate(method='linear')
    
    print(f"✅ Filled {points_filled:,} points from {len(short_outages)} short outages")
else:
    print("No short outages to fill")

# ============================================================================
# STEP 5: Apply Active Filter (Hz >= 55)
# ============================================================================

print("\n" + "="*80)
print("STEP 5: APPLYING ACTIVE FILTER (Hz >= 55)")
print("="*80)

df_active = df_processed[df_processed['Hz'] >= 55].copy()

removed_count = len(df_processed) - len(df_active)
removed_percentage = (removed_count / len(df_processed)) * 100

print(f"Original rows: {len(df_processed):,}")
print(f"Active rows (Hz >= 55): {len(df_active):,}")
print(f"Removed rows: {removed_count:,} ({removed_percentage:.2f}%)")

print("\nFinal Active Data Statistics:")
print(df_active['Hz'].describe())

# ============================================================================
# STEP 6: Matplotlib Comparison Plots
# ============================================================================

print("\n" + "="*80)
print("STEP 6: CREATING COMPARISON PLOTS")
print("="*80)

# Create figure with 3 subplots
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Plot 1: Raw Data
axes[0].plot(df_raw.index, df_raw['Hz'], linewidth=0.8, color='blue', alpha=0.7)
axes[0].axhline(y=55, color='red', linestyle='--', linewidth=2, label='Hz = 55 threshold')
axes[0].set_ylabel('Frequency (Hz)', fontsize=12)
axes[0].set_title('Raw Data (Original)', fontsize=14, fontweight='bold')
axes[0].legend(loc='upper right')
axes[0].grid(True, alpha=0.3)

# Plot 2: After Interpolation
axes[1].plot(df_processed.index, df_processed['Hz'], linewidth=0.8, color='orange', alpha=0.7)
axes[1].axhline(y=55, color='red', linestyle='--', linewidth=2, label='Hz = 55 threshold')
axes[1].set_ylabel('Frequency (Hz)', fontsize=12)
axes[1].set_title('After Interpolation (Short outages filled)', fontsize=14, fontweight='bold')
axes[1].legend(loc='upper right')
axes[1].grid(True, alpha=0.3)

# Plot 3: Final Active Data
axes[2].plot(df_active.index, df_active['Hz'], linewidth=0.8, color='green', alpha=0.7)
# axes[2].axhline(y=55, color='red', linestyle='--', linewidth=2, label='Hz = 55 threshold')
axes[2].set_xlabel('Timestamp', fontsize=12)
axes[2].set_ylabel('Frequency (Hz)', fontsize=12)
axes[2].set_title('Final Active Data (Hz >= 55 only)', fontsize=14, fontweight='bold')
axes[2].legend(loc='upper right')
axes[2].grid(True, alpha=0.3)

# Rotate x-axis labels for better readability
for ax in axes:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('data_preprocessing_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Comparison plot saved to: data_preprocessing_comparison.png")

# ============================================================================
# STEP 8: Summary Report
# ============================================================================

print("\n" + "="*80)
print("PREPROCESSING SUMMARY REPORT")
print("="*80)

summary = {
    'Stage': ['Raw Data', 'After Interpolation', 'Final Active Data'],
    'Data Points': [len(df_raw), len(df_processed), len(df_active)],
    'Mean Hz': [df_raw['Hz'].mean(), df_processed['Hz'].mean(), df_active['Hz'].mean()],
    'Std Hz': [df_raw['Hz'].std(), df_processed['Hz'].std(), df_active['Hz'].std()],
    'Min Hz': [df_raw['Hz'].min(), df_processed['Hz'].min(), df_active['Hz'].min()],
    'Max Hz': [df_raw['Hz'].max(), df_processed['Hz'].max(), df_active['Hz'].max()]
}

summary_df = pd.DataFrame(summary)
print("\n", summary_df.to_string(index=False))

# Save summary
summary_df.to_csv('preprocessing_summary.csv', index=False)
print("\n✅ Summary saved to: preprocessing_summary.csv")

# ============================================================================
# STEP 9: Save Processed Data
# ============================================================================

# print("\n" + "="*80)
# print("STEP 9: SAVING PROCESSED DATA")
# print("="*80)

# # Save final active data
# df_active.to_csv('V_0_processed_active.csv')
# print(f"✅ Active data saved to: V_0_processed_active.csv ({len(df_active):,} rows)")

# # Also save interpolated data (in case needed)
# df_processed.to_csv('V_0_processed_interpolated.csv')
# print(f"✅ Interpolated data saved to: V_0_processed_interpolated.csv ({len(df_processed):,} rows)")

# print("\n" + "="*80)
# print("✅ PREPROCESSING COMPLETE!")
# print("="*80)
# print(f"\nData is now ready for model training.")
# print(f"Use 'df_active' variable for training (shape: {df_active.shape})")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, RepeatVector, TimeDistributed, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler

# =========================================================
# CONFIGURATION (Fill from Optuna) trial 39
# =========================================================
SEQ_SIZE = 20
BATCH_SIZE = 64
EPOCHS = 40
LEARNING_RATE = 0.000549

LSTM_UNITS_1 = 256
LSTM_UNITS_2 = 64
DROPOUT_RATE = 0.1419


# =========================================================
# 1. SCALE FINAL ACTIVE DATA
# =========================================================

# Extract Hz values (shape: [N, 1])
hz_values = df_active[['Hz']].values.astype(np.float32)

# Min-Max Scaling (recommended for LSTM)
scaler = MinMaxScaler(feature_range=(0, 1))
hz_scaled = scaler.fit_transform(hz_values)

print("Scaled data shape:", hz_scaled.shape)


# =========================================================
# 2. CREATE SEQUENCES
# =========================================================

def create_sequences(data, seq_size):
    X = []
    for i in range(len(data) - seq_size):
        X.append(data[i:i + seq_size])
    return np.array(X)


X_train = create_sequences(hz_scaled, SEQ_SIZE)

# Autoencoder target = input
y_train = X_train.copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)


# =========================================================
# 3. BUILD LSTM AUTOENCODER
# =========================================================

def build_lstm(input_shape):

    model = Sequential()

    # ---------------- Encoder ----------------

    model.add(LSTM(
        LSTM_UNITS_1,
        activation='tanh',
        recurrent_activation='sigmoid',
        input_shape=input_shape,
        return_sequences=True
    ))
    model.add(Dropout(DROPOUT_RATE))


    model.add(LSTM(
        LSTM_UNITS_2,
        activation='tanh',
        recurrent_activation='sigmoid',
        return_sequences=False
    ))
    model.add(Dropout(DROPOUT_RATE))


    # ---------------- Bottleneck ----------------

    model.add(RepeatVector(input_shape[0]))


    # ---------------- Decoder ----------------

    model.add(LSTM(
        LSTM_UNITS_2,
        activation='tanh',
        recurrent_activation='sigmoid',
        return_sequences=True
    ))
    model.add(Dropout(DROPOUT_RATE))


    model.add(LSTM(
        LSTM_UNITS_1,
        activation='tanh',
        recurrent_activation='sigmoid',
        return_sequences=True
    ))
    model.add(Dropout(DROPOUT_RATE))


    # ---------------- Output ----------------

    model.add(TimeDistributed(Dense(input_shape[1])))


    # ---------------- Compile ----------------

    optimizer = Adam(learning_rate=LEARNING_RATE)

    model.compile(
        optimizer=optimizer,
        loss='mae',     
        metrics=['mape']
    )

    return model


# =========================================================
# 4. TRAIN MODEL
# =========================================================

input_shape = (SEQ_SIZE, X_train.shape[2])

model = build_lstm(input_shape)

model.summary()


history = model.fit(
    X_train,
    y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.2,
    shuffle=True,
    verbose=1
)


In [ ]:
# >>> ADD THIS BLOCK TO GET YOUR NUMBERS <<<
print("\n" + "="*50)
print("✅ COPY THESE VALUES TO YOUR INFERENCE SCRIPT")
print("="*50)
print(f"TRAIN_MIN = {scaler.data_min_[0]:.8f}")
print(f"TRAIN_MAX = {scaler.data_max_[0]:.8f}")
print("="*50 + "\n")

In [ ]:
# ============================================================
# LSTM AUTOENCODER - MULTI-THRESHOLD TRAINING EVALUATION
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("="*80)
print("LSTM AUTOENCODER: MULTI-THRESHOLD TRAINING EVALUATION")
print("="*80)

# ============================================================
# 1. PLOT TRAINING CURVES (MAE + MAPE)
# ============================================================
print("\n📊 Plotting Training Curves...")

# ---- MAE Loss ----
plt.figure(figsize=(10, 4))
plt.plot(history.history['loss'], label='Train MAE', linewidth=2)
plt.plot(history.history['val_loss'], label='Validation MAE', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('MAE', fontsize=12)
plt.title('LSTM Autoencoder - Training & Validation MAE', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_mae.png', dpi=300, bbox_inches='tight')
plt.show()

# ---- MAPE ----
plt.figure(figsize=(10, 4))
plt.plot(history.history['mape'], label='Train MAPE', linewidth=2)
plt.plot(history.history['val_mape'], label='Validation MAPE', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('MAPE (%)', fontsize=12)
plt.title('LSTM Autoencoder - Training & Validation MAPE', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('training_mape.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# 2. RECONSTRUCT TRAINING DATA
# ============================================================
print("\n🔄 Reconstructing Training Data...")
X_pred = model.predict(X_train, verbose=1)
print(f"Prediction shape: {X_pred.shape}")
print(f"Input shape     : {X_train.shape}")

# ============================================================
# 3. COMPUTE RECONSTRUCTION ERROR (MAE)
# ============================================================
print("\n📏 Computing Reconstruction Error...")
train_mae_loss = np.mean(
    np.abs(X_pred - X_train),
    axis=(1, 2)   # Average over sequence length and features
)

print(f"Reconstruction error vector shape: {train_mae_loss.shape}")
print(f"\nError Statistics:")
print(f"  Mean   : {train_mae_loss.mean():.6f}")
print(f"  Median : {np.median(train_mae_loss):.6f}")
print(f"  Std    : {train_mae_loss.std():.6f}")
print(f"  Min    : {train_mae_loss.min():.6f}")
print(f"  Max    : {train_mae_loss.max():.6f}")

# ============================================================
# 4. CALCULATE MULTIPLE THRESHOLDS
# ============================================================
print("\n" + "="*80)
print("🎯 CALCULATING MULTIPLE DETECTION THRESHOLDS")
print("="*80)

# Define percentile range to test
PERCENTILES = [99.1, 99.2, 99.3, 99.4, 99.5, 99.6, 99.7, 99.8, 99.9]

# Calculate thresholds for each percentile
thresholds = {}
threshold_stats = []

print(f"\n{'Percentile (%)':<15} {'Threshold':<15} {'Training FP Count':<20} {'Training FP Rate (%)':<20}")
print("-"*80)

for p in PERCENTILES:
    threshold = np.percentile(train_mae_loss, p)
    fp_count = np.sum(train_mae_loss > threshold)
    fp_rate = (fp_count / len(train_mae_loss)) * 100
    
    thresholds[p] = threshold
    threshold_stats.append({
        'percentile': p,
        'threshold': threshold,
        'fp_count': fp_count,
        'fp_rate': fp_rate
    })
    
    print(f"{p:<15.1f} {threshold:<15.6f} {fp_count:<20} {fp_rate:<20.4f}")

# Convert to DataFrame
thresholds_df = pd.DataFrame(threshold_stats)

# Save thresholds
thresholds_df.to_csv('percentile_thresholds.csv', index=False)
print("\n✅ Thresholds saved to: percentile_thresholds.csv")

# ============================================================
# 5. PLOT ERROR DISTRIBUTION WITH ALL THRESHOLDS
# ============================================================
print("\n📊 Plotting Error Distribution with Multiple Thresholds...")

plt.figure(figsize=(14, 6))

# Plot histogram
sns.histplot(
    train_mae_loss,
    bins=100,
    kde=True,
    stat="density",
    alpha=0.6,
    color='skyblue',
    edgecolor='black',
    linewidth=0.5
)

# Add threshold lines for key percentiles
key_percentiles = [99.1, 99.5, 99.6, 99.7, 99.9]
colors = ['green', 'blue', 'purple', 'orange', 'red']
linestyles = ['--', '--', '-', '--', '--']
linewidths = [1.5, 1.5, 2.5, 1.5, 1.5]

for p, color, ls, lw in zip(key_percentiles, colors, linestyles, linewidths):
    threshold = thresholds[p]
    label = f'{p}% = {threshold:.6f}'
    if p == 99.6:
        label = f'{p}% = {threshold:.6f} ⭐'  # Mark optimal
    
    plt.axvline(
        threshold,
        color=color,
        linestyle=ls,
        linewidth=lw,
        alpha=0.8,
        label=label
    )

plt.title("Reconstruction Error Distribution with Multiple Thresholds", 
          fontsize=14, fontweight='bold')
plt.xlabel("Reconstruction Error (MAE)", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.legend(loc='upper right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('error_distribution_multi_threshold.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# 6. PLOT THRESHOLD VS PERCENTILE ANALYSIS
# ============================================================
print("\n📈 Plotting Threshold Analysis...")

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Threshold vs Percentile
ax1 = axes[0, 0]
ax1.plot(thresholds_df['percentile'], thresholds_df['threshold'], 
         'b-o', linewidth=2, markersize=8)
# Highlight optimal threshold (99.6%)
optimal_idx = thresholds_df[thresholds_df['percentile'] == 99.6].index[0]
optimal_threshold = thresholds_df.loc[optimal_idx, 'threshold']
ax1.scatter(99.6, optimal_threshold, s=300, c='red', marker='*', 
           edgecolors='black', linewidths=2, zorder=5, label='Optimal (99.6%)')
ax1.set_xlabel('Percentile (%)', fontsize=12)
ax1.set_ylabel('Threshold Value', fontsize=12)
ax1.set_title('Threshold vs Percentile', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# Plot 2: False Positive Rate vs Percentile
ax2 = axes[0, 1]
ax2.plot(thresholds_df['percentile'], thresholds_df['fp_rate'], 
         'r-o', linewidth=2, markersize=8)
ax2.axvline(99.6, color='black', linestyle='--', linewidth=2, alpha=0.5, label='Optimal (99.6%)')
ax2.set_xlabel('Percentile (%)', fontsize=12)
ax2.set_ylabel('False Positive Rate on Training (%)', fontsize=12)
ax2.set_title('Training FP Rate vs Percentile', fontsize=14, fontweight='bold')
ax2.legend(fontsize=11)
ax2.grid(True, alpha=0.3)

# Plot 3: Error Over Time with Multiple Thresholds
ax3 = axes[1, 0]
ax3.plot(train_mae_loss, linewidth=0.8, alpha=0.7, color='blue', label='Reconstruction Error')

for p, color in zip(key_percentiles, colors):
    threshold = thresholds[p]
    label = f'{p}%'
    if p == 99.6:
        label = f'{p}% ⭐'
    ax3.axhline(threshold, color=color, linestyle='--', linewidth=2, 
               alpha=0.7, label=label)

ax3.set_xlabel('Sequence Index', fontsize=12)
ax3.set_ylabel('MAE', fontsize=12)
ax3.set_title('Reconstruction Error Over Time with Thresholds', fontsize=14, fontweight='bold')
ax3.legend(loc='upper right', fontsize=10)
ax3.grid(True, alpha=0.3)

# Plot 4: Zoom on Error Distribution (up to 99th percentile)
ax4 = axes[1, 1]
max_val = np.percentile(train_mae_loss, 99)
filtered_errors = train_mae_loss[train_mae_loss <= max_val]

ax4.hist(filtered_errors, bins=80, alpha=0.6, color='skyblue', edgecolor='black')
for p, color in zip(key_percentiles, colors):
    threshold = thresholds[p]
    ax4.axvline(threshold, color=color, linestyle='--', linewidth=2, alpha=0.8)

ax4.set_xlabel('Reconstruction Error (MAE)', fontsize=12)
ax4.set_ylabel('Frequency', fontsize=12)
ax4.set_title('Error Distribution (Zoomed to 99th Percentile)', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('threshold_analysis_complete.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================
# 7. IDENTIFY BEST THRESHOLD (For Future Testing)
# ============================================================
print("\n" + "="*80)
print("🏆 RECOMMENDED THRESHOLD FOR ANOMALY DETECTION")
print("="*80)

# Based on previous testing, 99.6% was optimal
OPTIMAL_PERCENTILE = 99.6
OPTIMAL_THRESHOLD = thresholds[OPTIMAL_PERCENTILE]

print(f"\n⭐ OPTIMAL PERCENTILE: {OPTIMAL_PERCENTILE}%")
print(f"⭐ OPTIMAL THRESHOLD: {OPTIMAL_THRESHOLD:.6f}")

optimal_stats = thresholds_df[thresholds_df['percentile'] == OPTIMAL_PERCENTILE].iloc[0]
print(f"\nStatistics at Optimal Threshold:")
print(f"  Training FP Count: {int(optimal_stats['fp_count'])}")
print(f"  Training FP Rate : {optimal_stats['fp_rate']:.4f}%")

print(f"\n💡 Use this threshold for FDIA detection on test data!")

# ============================================================
# 8. CREATE SUMMARY TABLE
# ============================================================
print("\n" + "="*80)
print("📋 THRESHOLD SUMMARY TABLE")
print("="*80)

print(f"\n{thresholds_df.to_string(index=False)}")

# ============================================================
# 9. SAVE OPTIMAL THRESHOLD FOR TESTING
# ============================================================
import json

threshold_config = {
    'optimal_percentile': OPTIMAL_PERCENTILE,
    'optimal_threshold': float(OPTIMAL_THRESHOLD),
    'train_mae_mean': float(train_mae_loss.mean()),
    'train_mae_median': float(np.median(train_mae_loss)),
    'train_mae_std': float(train_mae_loss.std()),
    'train_mae_min': float(train_mae_loss.min()),
    'train_mae_max': float(train_mae_loss.max()),
    'all_thresholds': {str(p): float(thresholds[p]) for p in PERCENTILES}
}

with open('optimal_threshold_config.json', 'w') as f:
    json.dump(threshold_config, f, indent=4)

print("\n✅ Optimal threshold config saved to: optimal_threshold_config.json")

# ============================================================
# 10. PLOT ZOOMED VIEW OF EACH THRESHOLD
# ============================================================
print("\n📊 Creating Individual Threshold Plots...")

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for idx, p in enumerate(PERCENTILES):
    ax = axes[idx]
    threshold = thresholds[p]
    
    # Plot histogram
    ax.hist(train_mae_loss, bins=80, alpha=0.6, color='skyblue', edgecolor='black')
    
    # Add threshold line
    ax.axvline(threshold, color='red', linestyle='--', linewidth=2.5, 
              label=f'Threshold = {threshold:.6f}')
    
    # Mark as optimal if 99.6%
    title = f'{p}% Percentile'
    if p == 99.6:
        title = f'{p}% Percentile ⭐ OPTIMAL'
        ax.set_facecolor('#fffef0')  # Light yellow background
    
    ax.set_xlabel('Reconstruction Error (MAE)', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('individual_thresholds.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================================
# 11. FINAL SUMMARY
# ============================================================================
print("\n" + "="*80)
print("✅ MULTI-THRESHOLD ANALYSIS COMPLETE!")
print("="*80)

print(f"\nFiles Created:")
print(f"  1. training_mae.png - Training curves (MAE)")
print(f"  2. training_mape.png - Training curves (MAPE)")
print(f"  3. error_distribution_multi_threshold.png - Distribution with all thresholds")
print(f"  4. threshold_analysis_complete.png - Comprehensive analysis")
print(f"  5. individual_thresholds.png - Individual threshold views")
print(f"  6. percentile_thresholds.csv - Threshold data")
print(f"  7. optimal_threshold_config.json - Configuration for testing")

print(f"\n🎯 RECOMMENDED FOR FDIA TESTING:")
print(f"   Percentile: {OPTIMAL_PERCENTILE}%")
print(f"   Threshold: {OPTIMAL_THRESHOLD:.6f}")

print("\n" + "="*80)

In [ ]:
# ==========================================
# Testing Script: LSTM Autoencoder (FDIA)
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc
)


# ==========================================
# CONFIG
# ==========================================

TEST_DATA_PATH = "V3S2.csv"

# >>> CONTROL SWITCH <<<
USE_DAYTIME_ONLY = False     # True = 6AM–6PM | False = Full 24h

DAY_START = 6
DAY_END = 18


# ==========================================
# THRESHOLD (From Training)
# ==========================================

THRESHOLD = 0.025

# ==========================================
# PRINT CONFIG
# ==========================================

print("="*70)
print("FDIA TESTING CONFIGURATION")
print("="*70)
print(f"Threshold        : {THRESHOLD:.6f}")
print(f"Daytime Only     : {USE_DAYTIME_ONLY}")

if USE_DAYTIME_ONLY:
    print(f"Daytime Window   : {DAY_START}:00 - {DAY_END}:00")
else:
    print("Using FULL 24h Dataset")

print("="*70)


# ==========================================
# LOAD DATA
# ==========================================

test_df = pd.read_csv(TEST_DATA_PATH)

test_df["datetimestamp"] = pd.to_datetime(
    test_df["datetimestamp"]
)

print("\nOriginal shape:", test_df.shape)


# ==========================================
# OPTIONAL DAYTIME FILTER
# ==========================================

if USE_DAYTIME_ONLY:

    test_df["hour"] = test_df["datetimestamp"].dt.hour

    test_df = test_df[
        (test_df["hour"] >= DAY_START) &
        (test_df["hour"] < DAY_END)
    ].copy()

    test_df.drop(columns=["hour"], inplace=True)

    print("After daytime filter:", test_df.shape)

else:

    print("No time filtering applied.")


# ==========================================
# PREPROCESSING
# ==========================================

# Signal
test_hz = test_df[["Hz_mod_anomaly"]].values.astype(np.float32)

# Labels
true_labels = (test_df["mod_BIN"] != 0).astype(int).values


# Use SAME scaler
test_hz_scaled = scaler.transform(test_hz)


# ==========================================
# CREATE SEQUENCES
# ==========================================

def create_sequences(data, seq_size):

    X = []

    for i in range(len(data) - seq_size):
        X.append(data[i:i + seq_size])

    return np.array(X)


X_test = create_sequences(test_hz_scaled, SEQ_SIZE)

true_labels_aligned = true_labels[SEQ_SIZE:]
timestamps = test_df["datetimestamp"].values[SEQ_SIZE:]


print("\nTest sequences:", X_test.shape)
print("Aligned labels:", true_labels_aligned.shape)


# ==========================================
# PREDICTION
# ==========================================

print("\nRunning inference...")

X_test_pred = model.predict(X_test, verbose=1)


# ==========================================
# RECONSTRUCTION ERROR
# ==========================================

test_mae = np.mean(
    np.abs(X_test_pred - X_test),
    axis=(1, 2)
)


print("\nMAE Statistics:")
print(f"Min   : {test_mae.min():.6f}")
print(f"Max   : {test_mae.max():.6f}")
print(f"Mean  : {test_mae.mean():.6f}")
print(f"Median: {np.median(test_mae):.6f}")


# ==========================================
# ANOMALY DETECTION
# ==========================================

predicted_labels = (test_mae > THRESHOLD).astype(int)


# ==========================================
# CONFUSION MATRIX
# ==========================================

conf_matrix = confusion_matrix(
    true_labels_aligned,
    predicted_labels
)

print("\nConfusion Matrix:")
print(conf_matrix)


if conf_matrix.shape == (2, 2):

    TN, FP, FN, TP = conf_matrix.ravel()

    accuracy = (TP + TN) / (TP + FP + TN + FN)

    precision = TP / (TP + FP) if (TP + FP) else 0

    recall = TP / (TP + FN) if (TP + FN) else 0

    f1 = (
        2 * precision * recall /
        (precision + recall)
        if (precision + recall) else 0
    )


    print("\n" + "="*50)
    print("DETECTION PERFORMANCE")
    print("="*50)

    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")

    print("="*50)
    print(f"TP: {TP} | FP: {FP}")
    print(f"FN: {FN} | TN: {TN}")
    print("="*50)


# ==========================================
# ROC CURVE
# ==========================================

print("\nComputing ROC Curve...")

fpr, tpr, _ = roc_curve(
    true_labels_aligned,
    test_mae
)

roc_auc = auc(fpr, tpr)


plt.figure(figsize=(6, 6))

plt.plot(
    fpr, tpr,
    linewidth=2,
    label=f"AUC = {roc_auc:.4f}"
)

plt.plot([0, 1], [0, 1], "k--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")

plt.title("ROC Curve - FDIA Detection")

plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print("AUC:", roc_auc)


# ==========================================
# VISUALIZATION
# ==========================================

fig, axes = plt.subplots(3, 1, figsize=(15, 12))


# -------- Hz --------

axes[0].plot(
    timestamps,
    test_df["Hz_mod_anomaly"].values[SEQ_SIZE:],
    linewidth=0.8,
    label="Hz"
)

anomaly_idx = np.where(predicted_labels == 1)[0]

axes[0].scatter(
    timestamps[anomaly_idx],
    test_df["Hz_mod_anomaly"].values[SEQ_SIZE:][anomaly_idx],
    color="red",
    s=12,
    label="Detected Attacks"
)

axes[0].set_title("Hz with Detected FDIA")
axes[0].set_ylabel("Frequency (Hz)")
axes[0].legend()
axes[0].grid(alpha=0.3)


# -------- MAE --------

axes[1].plot(
    timestamps,
    test_mae,
    linewidth=0.8,
    label="MAE"
)

axes[1].axhline(
    THRESHOLD,
    linestyle="--",
    color="red",
    label="Threshold"
)

axes[1].set_title("Reconstruction Error")
axes[1].set_ylabel("MAE")
axes[1].legend()
axes[1].grid(alpha=0.3)


# -------- True vs Pred --------

axes[2].fill_between(
    timestamps,
    0,
    true_labels_aligned,
    step="mid",
    alpha=0.3,
    label="True Attacks"
)

axes[2].fill_between(
    timestamps,
    0,
    predicted_labels,
    step="mid",
    alpha=0.3,
    label="Predicted Attacks"
)

axes[2].set_title("True vs Predicted FDIA")
axes[2].set_ylabel("State")

axes[2].set_yticks([0, 1])
axes[2].set_yticklabels(["Normal", "Attack"])

axes[2].legend()
axes[2].grid(alpha=0.3)


# Format X-axis
for ax in axes:
    ax.tick_params(axis="x", rotation=45)


plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Testing Script: LSTM Autoencoder (FDIA)
# Multi-Threshold Testing with Dataset Labeling
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from datetime import datetime

from sklearn.metrics import (
    confusion_matrix,
    roc_curve,
    auc,
    classification_report
)

# ==========================================
# CONFIG
# ==========================================

TEST_DATA_PATH = "V3S3.csv"

# Extract dataset name from filename (e.g., "V3S2" from "V3S2.csv")
DATASET_NAME = os.path.splitext(os.path.basename(TEST_DATA_PATH))[0]

# >>> CONTROL SWITCH <<<
USE_DAYTIME_ONLY = False     # True = 6AM–6PM | False = Full 24h

DAY_START = 6
DAY_END = 18

# ==========================================
# LOAD OPTIMAL THRESHOLD FROM TRAINING
# ==========================================

# Option 1: Load from JSON (if you saved it during training)
try:
    with open('optimal_threshold_config.json', 'r') as f:
        threshold_config = json.load(f)
    OPTIMAL_THRESHOLD = threshold_config['optimal_threshold']
    OPTIMAL_PERCENTILE = threshold_config['optimal_percentile']
    print(f"✅ Loaded optimal threshold from config: {OPTIMAL_THRESHOLD:.6f} ({OPTIMAL_PERCENTILE}%)")
except FileNotFoundError:
    # Option 2: Manual entry (use your training result)
    OPTIMAL_THRESHOLD = 0.009238  # ← Your 99.6th percentile value from training
    OPTIMAL_PERCENTILE = 99.6
    print(f"⚠️  Using manually set threshold: {OPTIMAL_THRESHOLD:.6f} ({OPTIMAL_PERCENTILE}%)")

# ==========================================
# MULTI-THRESHOLD TESTING (Optional)
# ==========================================

# Set to True to test multiple thresholds
TEST_MULTIPLE_THRESHOLDS = True

# Load all thresholds from training
try:
    thresholds_df = pd.read_csv('percentile_thresholds.csv')
    PERCENTILES = thresholds_df['percentile'].tolist()
    THRESHOLDS_DICT = dict(zip(thresholds_df['percentile'], thresholds_df['threshold']))
    print(f"✅ Loaded {len(PERCENTILES)} thresholds from training")
except FileNotFoundError:
    # Fallback: Use single optimal threshold
    TEST_MULTIPLE_THRESHOLDS = False
    PERCENTILES = [OPTIMAL_PERCENTILE]
    THRESHOLDS_DICT = {OPTIMAL_PERCENTILE: OPTIMAL_THRESHOLD}
    print("⚠️  Using single optimal threshold only")

# ==========================================
# PRINT CONFIG
# ==========================================

print("\n" + "="*70)
print(f"FDIA TESTING: {DATASET_NAME}")
print("="*70)
print(f"Test File        : {TEST_DATA_PATH}")
print(f"Dataset          : {DATASET_NAME}")
print(f"Optimal Threshold: {OPTIMAL_THRESHOLD:.6f} ({OPTIMAL_PERCENTILE}%)")
print(f"Daytime Only     : {USE_DAYTIME_ONLY}")
print(f"Sequence Size    : {SEQ_SIZE}")

if USE_DAYTIME_ONLY:
    print(f"Daytime Window   : {DAY_START}:00 - {DAY_END}:00")
else:
    print("Using FULL 24h Dataset")

if TEST_MULTIPLE_THRESHOLDS:
    print(f"\nMulti-Threshold Testing: ENABLED")
    print(f"Thresholds to test:")
    for p in PERCENTILES:
        marker = " ⭐" if p == OPTIMAL_PERCENTILE else ""
        print(f" {p:>5.1f}% -> {THRESHOLDS_DICT[p]:.6f}{marker}")
else:
    print(f"\nMulti-Threshold Testing: DISABLED")

print("="*70)

# ==========================================
# LOAD DATA
# ==========================================

print(f"\n📂 Loading test data: {DATASET_NAME}...")
test_df = pd.read_csv(TEST_DATA_PATH)

test_df["datetimestamp"] = pd.to_datetime(test_df["datetimestamp"])

print(f"Original shape: {test_df.shape}")

# ==========================================
# OPTIONAL DAYTIME FILTER
# ==========================================

if USE_DAYTIME_ONLY:
    test_df["hour"] = test_df["datetimestamp"].dt.hour
    test_df = test_df[
        (test_df["hour"] >= DAY_START) &
        (test_df["hour"] < DAY_END)
    ].copy()
    test_df.drop(columns=["hour"], inplace=True)
    print(f"After daytime filter: {test_df.shape}")
else:
    print("No time filtering applied.")

# ==========================================
# PREPROCESSING
# ==========================================

print("\n⚙️  Preprocessing...")

# Signal
test_hz = test_df[["Hz_mod_anomaly"]].values.astype(np.float32)

# Labels
true_labels = (test_df["mod_BIN"] != 0).astype(int).values

# Use SAME scaler from training
test_hz_scaled = scaler.transform(test_hz)

# ==========================================
# CREATE SEQUENCES
# ==========================================

def create_sequences(data, seq_size):
    X = []
    for i in range(len(data) - seq_size):
        X.append(data[i:i + seq_size])
    return np.array(X)

X_test = create_sequences(test_hz_scaled, SEQ_SIZE)

true_labels_aligned = true_labels[SEQ_SIZE:]
timestamps = test_df["datetimestamp"].values[SEQ_SIZE:]

print(f"\nTest sequences : {X_test.shape}")
print(f"Aligned labels : {true_labels_aligned.shape}")

# ==========================================
# PREDICTION
# ==========================================

print("\n🔮 Running inference...")
X_test_pred = model.predict(X_test, verbose=1)

# ==========================================
# RECONSTRUCTION ERROR (MAE)
# ==========================================

print("\n📏 Computing reconstruction errors...")
test_mae = np.mean(
    np.abs(X_test_pred - X_test),
    axis=(1, 2)
)

print(f"\nMAE Statistics:")
print(f" Min   : {test_mae.min():.6f}")
print(f" Max   : {test_mae.max():.6f}")
print(f" Mean  : {test_mae.mean():.6f}")
print(f" Median: {np.median(test_mae):.6f}")

# ==========================================
# MULTI-THRESHOLD PERFORMANCE EVALUATION
# ==========================================

if TEST_MULTIPLE_THRESHOLDS:
    print("\n" + "="*70)
    print(f"MULTI-THRESHOLD PERFORMANCE - {DATASET_NAME}")
    print("="*70)
    
    results = []
    
    print(f"\n{'Percentile':<12} {'Threshold':<12} {'Accuracy':<10} {'Precision':<11} "
          f"{'Recall':<10} {'F1':<10} {'TP':<6} {'FP':<6} {'FN':<6}")
    print("-"*100)
    
    for p in PERCENTILES:
        threshold = THRESHOLDS_DICT[p]
        
        # Predict
        predicted = (test_mae > threshold).astype(int)
        
        # Confusion matrix
        tn, fp, fn, tp = confusion_matrix(true_labels_aligned, predicted).ravel()
        
        # Metrics
        accuracy = (tp + tn) / (tp + fp + tn + fn)
        precision = tp / (tp + fp) if (tp + fp) else 0
        recall = tp / (tp + fn) if (tp + fn) else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
        
        results.append({
            'dataset': DATASET_NAME,
            'percentile': p,
            'threshold': threshold,
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'tp': tp,
            'fp': fp,
            'fn': fn,
            'tn': tn
        })
        
        marker = " ⭐" if p == OPTIMAL_PERCENTILE else ""
        print(f"{p:<12.1f} {threshold:<12.6f} {accuracy:<10.4f} "
              f"{precision:<11.4f} {recall:<10.4f} {f1:<10.4f} "
              f"{tp:<6} {fp:<6} {fn:<6}{marker}")
    
    results_df = pd.DataFrame(results)
    
    # Save with dataset name in filename
    results_filename = f'test_results_multi_threshold_{DATASET_NAME}.csv'
    results_df.to_csv(results_filename, index=False)
    print(f"\n✅ Results saved to: {results_filename}")

# ==========================================
# OPTIMAL THRESHOLD PERFORMANCE
# ==========================================

print("\n" + "="*70)
print(f"PERFORMANCE WITH OPTIMAL THRESHOLD ({OPTIMAL_PERCENTILE}%) - {DATASET_NAME}")
print("="*70)

predicted_labels = (test_mae > OPTIMAL_THRESHOLD).astype(int)

# Confusion Matrix
conf_matrix = confusion_matrix(true_labels_aligned, predicted_labels)

print("\nConfusion Matrix:")
print(conf_matrix)

if conf_matrix.shape == (2, 2):
    TN, FP, FN, TP = conf_matrix.ravel()
    
    accuracy = (TP + TN) / (TP + FP + TN + FN)
    precision = TP / (TP + FP) if (TP + FP) else 0
    recall = TP / (TP + FN) if (TP + FN) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    
    print("\n" + "="*50)
    print("DETECTION PERFORMANCE")
    print("="*50)
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1-score : {f1:.4f}")
    print("="*50)
    print(f"TP: {TP} | FP: {FP}")
    print(f"FN: {FN} | TN: {TN}")
    print("="*50)

# ==========================================
# ROC CURVE
# ==========================================

print("\n📊 Computing ROC Curve...")

fpr, tpr, _ = roc_curve(true_labels_aligned, test_mae)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2.5, label=f"AUC = {roc_auc:.4f}")
plt.plot([0, 1], [0, 1], "k--", linewidth=1.5, alpha=0.5)
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate", fontsize=12)
plt.title(f"ROC Curve - FDIA Detection\nDataset: {DATASET_NAME}", 
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()

# Save with dataset name
roc_filename = f'roc_curve_{DATASET_NAME}.png'
plt.savefig(roc_filename, dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ ROC curve saved to: {roc_filename}")
print(f"AUC: {roc_auc:.4f}")

# ==========================================
# MULTI-THRESHOLD VISUALIZATION
# ==========================================

if TEST_MULTIPLE_THRESHOLDS:
    print(f"\n📊 Creating multi-threshold visualizations for {DATASET_NAME}...")
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Add dataset name to figure
    fig.suptitle(f'Multi-Threshold Performance Analysis - {DATASET_NAME}', 
                 fontsize=16, fontweight='bold', y=0.995)
    
    # Plot 1: F1-Score vs Percentile
    ax1 = axes[0, 0]
    ax1.plot(results_df['percentile'], results_df['f1'], 'b-o', linewidth=2, markersize=8)
    optimal_idx = results_df[results_df['percentile'] == OPTIMAL_PERCENTILE].index[0]
    ax1.scatter(OPTIMAL_PERCENTILE, results_df.loc[optimal_idx, 'f1'], 
               s=300, c='red', marker='*', edgecolors='black', linewidths=2, zorder=5)
    ax1.set_xlabel('Percentile (%)', fontsize=12)
    ax1.set_ylabel('F1-Score', fontsize=12)
    ax1.set_title(f'F1-Score vs Percentile\n{DATASET_NAME}', fontsize=13, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Precision & Recall vs Percentile
    ax2 = axes[0, 1]
    ax2.plot(results_df['percentile'], results_df['precision'], 'b-o', 
            linewidth=2, label='Precision', markersize=6)
    ax2.plot(results_df['percentile'], results_df['recall'], 'r-s', 
            linewidth=2, label='Recall', markersize=6)
    ax2.axvline(OPTIMAL_PERCENTILE, color='black', linestyle='--', alpha=0.5)
    ax2.set_xlabel('Percentile (%)', fontsize=12)
    ax2.set_ylabel('Score', fontsize=12)
    ax2.set_title(f'Precision & Recall vs Percentile\n{DATASET_NAME}', 
                  fontsize=13, fontweight='bold')
    ax2.legend(fontsize=11)
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([0, 1.05])
    
    # Plot 3: Attack Detection Rate
    ax3 = axes[1, 0]
    total_attacks = TP + FN
    ax3.plot(results_df['percentile'], results_df['tp'], 'g-o', 
            linewidth=2, label='Detected', markersize=6)
    ax3.plot(results_df['percentile'], results_df['fn'], 'r-s', 
            linewidth=2, label='Missed', markersize=6)
    ax3.axhline(total_attacks, color='black', linestyle='--', 
               linewidth=1, alpha=0.5, label=f'Total: {total_attacks}')
    ax3.axvline(OPTIMAL_PERCENTILE, color='black', linestyle='--', alpha=0.5)
    ax3.set_xlabel('Percentile (%)', fontsize=12)
    ax3.set_ylabel('Number of Attacks', fontsize=12)
    ax3.set_title(f'Attack Detection vs Percentile\n{DATASET_NAME}', 
                  fontsize=13, fontweight='bold')
    ax3.legend(fontsize=11)
    ax3.grid(True, alpha=0.3)
    
    # Plot 4: Precision-Recall Trade-off
    ax4 = axes[1, 1]
    ax4.plot(results_df['recall'], results_df['precision'], 'b-o', 
            linewidth=2, markersize=6)
    ax4.scatter(results_df.loc[optimal_idx, 'recall'], 
               results_df.loc[optimal_idx, 'precision'],
               s=300, c='red', marker='*', edgecolors='black', 
               linewidths=2, zorder=5, label=f'Optimal ({OPTIMAL_PERCENTILE}%)')
    ax4.set_xlabel('Recall', fontsize=12)
    ax4.set_ylabel('Precision', fontsize=12)
    ax4.set_title(f'Precision-Recall Trade-off\n{DATASET_NAME}', 
                  fontsize=13, fontweight='bold')
    ax4.legend(fontsize=11)
    ax4.grid(True, alpha=0.3)
    ax4.set_xlim([0, 1.05])
    ax4.set_ylim([0, 1.05])
    
    plt.tight_layout()
    
    # Save with dataset name
    perf_filename = f'multi_threshold_performance_{DATASET_NAME}.png'
    plt.savefig(perf_filename, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"✅ Performance plots saved to: {perf_filename}")

# ==========================================
# VISUALIZATION (Using Optimal Threshold)
# ==========================================

print(f"\n📊 Creating detection visualizations for {DATASET_NAME}...")

fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Add dataset name to figure
fig.suptitle(f'FDIA Detection Results - {DATASET_NAME}', 
             fontsize=16, fontweight='bold', y=0.995)

# -------- Hz with Detected Attacks --------
axes[0].plot(
    timestamps,
    test_df["Hz_mod_anomaly"].values[SEQ_SIZE:],
    linewidth=0.8,
    label="Hz",
    alpha=0.7,
    color='blue'
)

anomaly_idx = np.where(predicted_labels == 1)[0]
axes[0].scatter(
    timestamps[anomaly_idx],
    test_df["Hz_mod_anomaly"].values[SEQ_SIZE:][anomaly_idx],
    color="red",
    s=15,
    label="Detected Attacks",
    alpha=0.7
)

axes[0].set_title(f"Hz with Detected FDIA (Threshold: {OPTIMAL_THRESHOLD:.6f}, {OPTIMAL_PERCENTILE}%)", 
                  fontsize=13, fontweight='bold')
axes[0].set_ylabel("Frequency (Hz)", fontsize=12)
axes[0].legend(fontsize=11, loc='upper right')
axes[0].grid(alpha=0.3)

# Add text box with dataset info
textstr = f'Dataset: {DATASET_NAME}\nSeq Size: {SEQ_SIZE}'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.5)
axes[0].text(0.02, 0.98, textstr, transform=axes[0].transAxes, fontsize=10,
            verticalalignment='top', bbox=props)

# -------- Reconstruction Error (MAE) --------
axes[1].plot(
    timestamps,
    test_mae,
    linewidth=0.8,
    label="Reconstruction Error (MAE)",
    alpha=0.7,
    color='green'
)

axes[1].axhline(
    OPTIMAL_THRESHOLD,
    linestyle="--",
    color="red",
    linewidth=2,
    label=f"Threshold ({OPTIMAL_PERCENTILE}%): {OPTIMAL_THRESHOLD:.6f}"
)

axes[1].set_title("Reconstruction Error Over Time", fontsize=13, fontweight='bold')
axes[1].set_ylabel("MAE", fontsize=12)
axes[1].legend(fontsize=11, loc='upper right')
axes[1].grid(alpha=0.3)

# -------- True vs Predicted --------
axes[2].fill_between(
    timestamps,
    0,
    true_labels_aligned,
    step="mid",
    alpha=0.4,
    color='blue',
    label="True Attacks"
)

axes[2].fill_between(
    timestamps,
    0,
    predicted_labels,
    step="mid",
    alpha=0.4,
    color='red',
    label="Predicted Attacks"
)

axes[2].set_title(f"True vs Predicted FDIA", fontsize=13, fontweight='bold')
axes[2].set_xlabel("Timestamp", fontsize=12)
axes[2].set_ylabel("State", fontsize=12)
axes[2].set_yticks([0, 1])
axes[2].set_yticklabels(["Normal", "Attack"])
axes[2].legend(fontsize=11, loc='upper right')
axes[2].grid(alpha=0.3)

# Add performance metrics
perf_text = f'Accuracy: {accuracy:.3f}\nPrecision: {precision:.3f}\nRecall: {recall:.3f}\nF1: {f1:.3f}'
axes[2].text(0.02, 0.98, perf_text, transform=axes[2].transAxes, fontsize=10,
            verticalalignment='top', bbox=props)

# Format X-axis
for ax in axes:
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()

# Save with dataset name
detection_filename = f'fdia_detection_results_{DATASET_NAME}.png'
plt.savefig(detection_filename, dpi=300, bbox_inches='tight')
plt.show()

print(f"✅ Detection plots saved to: {detection_filename}")

# ==========================================
# SAVE FINAL RESULTS
# ==========================================

print("\n💾 Saving final results...")

final_results = {
    'dataset': DATASET_NAME,
    'test_file': TEST_DATA_PATH,
    'timestamp': datetime.now().isoformat(),
    'optimal_percentile': OPTIMAL_PERCENTILE,
    'optimal_threshold': float(OPTIMAL_THRESHOLD),
    'performance': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'auc': float(roc_auc)
    },
    'confusion_matrix': {
        'TP': int(TP),
        'FP': int(FP),
        'FN': int(FN),
        'TN': int(TN)
    },
    'test_mae_stats': {
        'mean': float(test_mae.mean()),
        'median': float(np.median(test_mae)),
        'std': float(test_mae.std()),
        'min': float(test_mae.min()),
        'max': float(test_mae.max())
    },
    'data_info': {
        'total_samples': int(len(test_df)),
        'total_sequences': int(len(X_test)),
        'total_attacks': int(TP + FN),
        'daytime_only': USE_DAYTIME_ONLY
    }
}

# Save with dataset name
results_json_filename = f'test_results_optimal_{DATASET_NAME}.json'
with open(results_json_filename, 'w') as f:
    json.dump(final_results, f, indent=4)

print(f"✅ Results saved to: {results_json_filename}")

# ==========================================
# SUMMARY
# ==========================================

print("\n" + "="*70)
print(f"✅ TESTING COMPLETE - {DATASET_NAME}")
print("="*70)

print(f"\n📊 Dataset: {DATASET_NAME}")
print(f"   Test File: {TEST_DATA_PATH}")

print(f"\n🎯 Optimal Threshold ({OPTIMAL_PERCENTILE}%):")
print(f"   Threshold : {OPTIMAL_THRESHOLD:.6f}")
print(f"   Accuracy  : {accuracy:.4f}")
print(f"   Precision : {precision:.4f}")
print(f"   Recall    : {recall:.4f}")
print(f"   F1-Score  : {f1:.4f}")
print(f"   AUC       : {roc_auc:.4f}")

print(f"\n📊 Detection Summary:")
print(f"   Total Attacks: {TP + FN}")
print(f"   Detected     : {TP} ({recall*100:.1f}%)")
print(f"   Missed       : {FN} ({(1-recall)*100:.1f}%)")
print(f"   False Alarms : {FP}")

print(f"\n📁 Files Created:")
if TEST_MULTIPLE_THRESHOLDS:
    print(f"   • test_results_multi_threshold_{DATASET_NAME}.csv")
    print(f"   • multi_threshold_performance_{DATASET_NAME}.png")
print(f"   • roc_curve_{DATASET_NAME}.png")
print(f"   • fdia_detection_results_{DATASET_NAME}.png")
print(f"   • test_results_optimal_{DATASET_NAME}.json")

print("\n" + "="*70)

In [ ]:
# >>> ADD THIS BLOCK TO GET YOUR NUMBERS <<<
print("\n" + "="*50)
print("✅ COPY THESE VALUES TO YOUR INFERENCE SCRIPT")
print("="*50)
print(f"TRAIN_MIN = {scaler.data_min_[0]:.8f}")
print(f"TRAIN_MAX = {scaler.data_max_[0]:.8f}")
print("="*50 + "\n")